# 01 — EDA & Business Insights

Exploratory analysis of the Kaggle Playground Series S6E3 telecom churn dataset. Goal: understand churn drivers and translate patterns into retention actions.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

%matplotlib inline

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from src.config import load_config, resolve_path
from src.data import load_raw_data, clean_total_charges, add_churn_binary

sns.set_theme(style="whitegrid")
config = load_config()
FIG_DIR = resolve_path(config, "metrics").parent / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

## Load data

In [ ]:
df_train, df_test = load_raw_data()
df = clean_total_charges(df_train)
df = add_churn_binary(df)

print("Shape:", df.shape)
print("\nMissing values after cleaning:")
print(df.isnull().sum().sort_values(ascending=False).head(10))
df.head()

## Overall churn rate

In [ ]:
churn_rate = df["Churn_binary"].mean()
print(f"Overall churn rate: {churn_rate:.1%}")

fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(x="Churn", data=df, ax=ax, palette="Set2")
ax.set_title("Churn distribution")
plt.tight_layout()
plt.savefig(FIG_DIR / "churn_distribution.png", dpi=120)
plt.show()

## Segment analysis — contract, tenure, internet, payment

In [ ]:
def plot_churn_by(col, rotation=0):
    rates = df.groupby(col)["Churn_binary"].mean().sort_values()
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.barplot(x=rates.index.astype(str), y=rates.values, ax=ax, palette="Blues_d")
    ax.set_title(f"Churn rate by {col}")
    ax.set_ylabel("Churn rate")
    ax.set_xlabel(col)
    plt.xticks(rotation=rotation)
    plt.tight_layout()
    return fig

for col, rot in [("Contract", 0), ("InternetService", 0), ("PaymentMethod", 45)]:
    fig = plot_churn_by(col, rot)
    fig.savefig(FIG_DIR / f"churn_by_{col.lower()}.png", dpi=120)
    plt.show()

In [ ]:
df["tenure_bucket"] = pd.cut(
    df["tenure"], bins=[0, 6, 12, 24, 60, 100], labels=["0-6", "6-12", "12-24", "24-60", "60+"]
)
fig = plot_churn_by("tenure_bucket")
fig.savefig(FIG_DIR / "churn_by_tenure.png", dpi=120)
plt.show()

## Numeric features vs churn

In [ ]:
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, numeric_cols):
    sns.boxplot(x="Churn", y=col, data=df, ax=ax)
    ax.set_title(f"{col} vs Churn")
plt.tight_layout()
plt.savefig(FIG_DIR / "numeric_vs_churn.png", dpi=120)
plt.show()

df[numeric_cols + ["SeniorCitizen"]].describe()

## Correlation heatmap

In [ ]:
corr = df[numeric_cols + ["SeniorCitizen", "Churn_binary"]].corr()
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation matrix")
plt.tight_layout()
plt.savefig(FIG_DIR / "correlation_heatmap.png", dpi=120)
plt.show()

## Actionable business findings

1. **Month-to-month contracts are the highest-risk segment** — churn is roughly 4× higher than annual contracts and 40× higher than two-year plans. Retention offers should prioritize contract upgrades in the first year.
2. **Fiber optic customers churn far more than DSL or phone-only** — likely price sensitivity and competitive alternatives. Bundle value-add services (security, support) for fiber subscribers.
3. **Electronic check payers show the highest churn** among payment methods — friction and lack of autopay correlate with disengagement. Incentivize automatic bank/card payments.
4. **Early tenure is fragile** — customers in the first 6–12 months show elevated churn. Proactive onboarding and check-ins during this window have outsized ROI.
5. **Higher monthly charges amplify risk when paired with short tenure** — price increases without demonstrated value drive exits. Target discount or loyalty bundles for high-ARPU, low-tenure accounts.